# Advanced: Multi-Strategy Missing Value Imputation with PAMOLA.CORE

**Goal**: Master all 3 imputation strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Statistical imputation (mean, median, mode)
- **Strategy 2**: Advanced methods (interpolation, random sampling)
- **Strategy 3**: Custom invalid value handling
- Calculate data quality metrics
- Analyze distribution impact
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of statistical imputation
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── transformations/
        └── 02_impute_missing_values_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display

print("🔍 Detecting PAMOLA installation...\n")

# Auto-detect project root (search up 6 levels for pamola_core/)
current_dir = Path.cwd()
project_root = current_dir
pamola_found = False

for level in range(6):
    if (project_root / 'pamola_core').exists():
        print(f"✓ Found pamola_core at level {level}: {project_root}")
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break
    project_root = project_root.parent

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.transformations.imputation.impute_missing_values import ImputeMissingValuesOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 8 columns)
- Sample records (first 5 rows)
- Column statistics (types, ranges, unique counts)
- Missing value summary per field

**Dataset features:**
- Multiple data types: numeric, categorical, date, text
- Realistic missing patterns (~20-30% per field)
- Invalid values mixed with genuine missing data

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'imputation_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate base data
    df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'age': np.random.randint(18, 80, n_records),
        'salary': np.random.randint(30000, 150000, n_records),
        'experience_years': np.random.randint(0, 40, n_records),
        'department': np.random.choice(['Sales', 'IT', 'HR', 'Finance', 'Operations'], n_records),
        'location': np.random.choice(['North', 'South', 'East', 'West', 'Central'], n_records),
        'performance_score': np.random.uniform(1.0, 5.0, n_records).round(2),
        'last_promotion_date': pd.date_range(start='2020-01-01', periods=n_records, freq='D')[:n_records]
    })
    
    # Introduce missing values (20-30% per field)
    missing_mask_age = np.random.random(n_records) < 0.25
    missing_mask_salary = np.random.random(n_records) < 0.25
    missing_mask_experience = np.random.random(n_records) < 0.20
    missing_mask_dept = np.random.random(n_records) < 0.20
    missing_mask_location = np.random.random(n_records) < 0.15
    missing_mask_score = np.random.random(n_records) < 0.30
    missing_mask_date = np.random.random(n_records) < 0.25
    
    df.loc[missing_mask_age, 'age'] = np.nan
    df.loc[missing_mask_salary, 'salary'] = np.nan
    df.loc[missing_mask_experience, 'experience_years'] = np.nan
    df.loc[missing_mask_dept, 'department'] = np.nan
    df.loc[missing_mask_location, 'location'] = np.nan
    df.loc[missing_mask_score, 'performance_score'] = np.nan
    df.loc[missing_mask_date, 'last_promotion_date'] = pd.NaT
    
    # Add some invalid values (5% of remaining records)
    invalid_mask = np.random.random(n_records) < 0.05
    df.loc[invalid_mask & ~missing_mask_age, 'age'] = -1
    df.loc[invalid_mask & ~missing_mask_salary, 'salary'] = 0
    df.loc[invalid_mask & ~missing_mask_experience, 'experience_years'] = -999
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    if col == 'record_id':
        continue
    dtype_str = str(df[col].dtype)
    missing_count = df[col].isna().sum()
    missing_pct = (missing_count / len(df)) * 100
    if pd.api.types.is_numeric_dtype(df[col]):
        non_null = df[col].dropna()
        if len(non_null) > 0:
            print(f"  {col:25s} ({dtype_str:10s}): min={non_null.min():.0f}, max={non_null.max():.0f}, missing={missing_count} ({missing_pct:.1f}%)")
    else:
        print(f"  {col:25s} ({dtype_str:10s}): {df[col].nunique()} unique, missing={missing_count} ({missing_pct:.1f}%)")

print(f"\n💡 Perfect dataset for testing all 3 strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field assignments for each strategy
   - `strategy1_fields`: Numeric fields for statistical methods
   - `strategy2_fields`: Fields for advanced methods
   - `strategy3_fields`: Fields with invalid values
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each strategy)
- Missing value counts per field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Strategy assignments:**
- Strategy 1: Statistical (age, salary, performance_score)
- Strategy 2: Advanced (experience_years, last_promotion_date)
- Strategy 3: Invalid handling (age, salary, experience_years)

**Note:** All fields must exist in dataset before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_fields": {
        "age": {"data_type": "numeric", "imputation_strategy": "median"},
        "salary": {"data_type": "numeric", "imputation_strategy": "mean"},
        "performance_score": {"data_type": "numeric", "imputation_strategy": "mean"}
    },
    "strategy2_fields": {
        "experience_years": {"data_type": "numeric", "imputation_strategy": "interpolation"},
        "department": {"data_type": "categorical", "imputation_strategy": "most_frequent"},
        "last_promotion_date": {"data_type": "date", "imputation_strategy": "median_date"}
    },
    "strategy3_fields": {
        "age": {"data_type": "numeric", "imputation_strategy": "median"},
        "salary": {"data_type": "numeric", "imputation_strategy": "mean"},
        "experience_years": {"data_type": "numeric", "imputation_strategy": "mean"}
    }
}

# Invalid values configuration for Strategy 3
INVALID_VALUES_CONFIG = {
    "age": [-1, 0, 999],
    "salary": [0, -1],
    "experience_years": [-999, -1]
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
all_fields = set()
for strategy, fields in FIELD_CONFIG.items():
    for field_name in fields.keys():
        all_fields.add(field_name)
        if field_name not in df.columns:
            raise ValueError(
                f"❌ Field '{field_name}' for {strategy} not found!\n"
                f"Available columns: {', '.join(df.columns)}\n"
                f"Please update FIELD_CONFIG"
            )

print("Strategy Field Validation:")
for strategy, fields in FIELD_CONFIG.items():
    print(f"\n  {strategy}:")
    for field_name, config in fields.items():
        missing_count = df[field_name].isna().sum()
        strategy_name = config.get('imputation_strategy', 'N/A')
        print(f"    ✓ {field_name:25s}: {strategy_name:15s} ({missing_count} missing)")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy",
    description="Multi-strategy missing value imputation",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Statistical Imputation

**How to use:**
- Uses statistical measures for numeric fields
- Applies mean/median based on data distribution
- Run to execute statistical strategy

**Key parameters:**
- `field_strategies`: age (median), salary (mean), performance_score (mean)
- `invalid_values=None`: Only handles genuine missing values
- `mode='ENRICH'`: Keeps original + adds imputed field

**What you'll see:**
- Configuration summary (3 fields)
- Progress bar: validation → statistics → imputation → metrics → viz → save
- Completion message with execution time (1-3 seconds)
- Missing value reduction per field

**Validate:**
- ✅ Execution time <5 seconds
- ✅ Zero missing values after imputation
- ✅ Distribution preserved (minimal skew change)
- ✅ Status = "completed"

**Note:** Median used for age (skewed), mean for salary/score (normal distribution)

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: STATISTICAL IMPUTATION")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Statistical",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = ImputeMissingValuesOperation(
    # Core parameters
    field_strategies=FIELD_CONFIG["strategy1_fields"],
    invalid_values=None,  # Only handle genuine missing values
    mode='ENRICH',  # Keep original + add new field
    
    # Output configuration
    column_prefix='imputed_',
    output_format='csv',
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  Fields: {len(FIELD_CONFIG['strategy1_fields'])}")
print(f"  Methods: {', '.join([v['imputation_strategy'] for v in FIELD_CONFIG['strategy1_fields'].values()])}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_statistical',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_statistical' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    
    print(f"\n📊 Missing Values Reduction:")
    for field_name in FIELD_CONFIG["strategy1_fields"].keys():
        output_field = f"imputed_{field_name}"
        if output_field in result_df_s1.columns:
            before = df[field_name].isna().sum()
            after = result_df_s1[output_field].isna().sum()
            print(f"  {field_name:25s}: {before} → {after} missing")

## STRATEGY 2: Advanced Methods

**How to use:**
- Uses interpolation for numeric sequences
- Applies most_frequent for categorical data
- Handles date/time imputation
- Run to execute advanced strategy

**Key parameters:**
- `field_strategies`: experience_years (interpolation), department (most_frequent), last_promotion_date (median_date)
- `invalid_values=None`: Only handles genuine missing values
- `mode='ENRICH'`: Keeps original + adds imputed field

**What you'll see:**
- Configuration summary (3 fields)
- Progress bar: validation → interpolation → imputation → metrics → viz → save
- Completion message with execution time (1-3 seconds)
- Missing value reduction per field

**Validate:**
- ✅ Execution time <5 seconds
- ✅ Smooth interpolation (no jumps)
- ✅ Mode preserved for categorical
- ✅ Status = "completed"

**Note:** Interpolation ideal for time-series or sequential data; most_frequent for categorical maintains distribution

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: ADVANCED METHODS")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Advanced",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = ImputeMissingValuesOperation(
    field_strategies=FIELD_CONFIG["strategy2_fields"],
    invalid_values=None,
    mode='ENRICH',
    column_prefix='imputed_',
    output_format='csv',
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  Fields: {len(FIELD_CONFIG['strategy2_fields'])}")
print(f"  Methods: {', '.join([v['imputation_strategy'] for v in FIELD_CONFIG['strategy2_fields'].values()])}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_advanced',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results (NEWEST file)
output_files_s2 = sorted(
    list((task_dir / 'strategy2_advanced' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    
    print(f"\n📊 Missing Values Reduction:")
    for field_name in FIELD_CONFIG["strategy2_fields"].keys():
        output_field = f"imputed_{field_name}"
        if output_field in result_df_s2.columns:
            before = df[field_name].isna().sum()
            after = result_df_s2[output_field].isna().sum()
            print(f"  {field_name:25s}: {before} → {after} missing")

## STRATEGY 3: Invalid Value Handling

**How to use:**
- Treats specified values as missing (e.g., -1, 0, -999)
- Imputes both genuine missing and invalid values
- Run to execute invalid value strategy

**Key parameters:**
- `field_strategies`: age (median), salary (mean), experience_years (mean)
- `invalid_values`: Custom dict mapping fields to invalid values
- `mode='ENRICH'`: Keeps original + adds imputed field

**What you'll see:**
- Configuration summary (3 fields with invalid value lists)
- Progress bar: validation → detection → imputation → metrics → viz → save
- Completion message with execution time (1-3 seconds)
- Combined missing + invalid value reduction

**Invalid value examples:**
- age: -1, 0, 999 (placeholder values)
- salary: 0, -1 (data entry errors)
- experience_years: -999, -1 (sentinel values)

**Validate:**
- ✅ Execution time <5 seconds
- ✅ All invalid values replaced
- ✅ Distribution remains realistic
- ✅ Status = "completed"

**Note:** Critical for cleaning datasets with known data quality issues

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: INVALID VALUE HANDLING")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Invalid Values",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = ImputeMissingValuesOperation(
    field_strategies=FIELD_CONFIG["strategy3_fields"],
    invalid_values=INVALID_VALUES_CONFIG,  # Treat specified values as missing
    mode='ENRICH',
    column_prefix='imputed_',
    output_format='csv',
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  Fields: {len(FIELD_CONFIG['strategy3_fields'])}")
print(f"  Invalid values defined for: {', '.join(INVALID_VALUES_CONFIG.keys())}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_invalid',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results (NEWEST file)
output_files_s3 = sorted(
    list((task_dir / 'strategy3_invalid' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    
    print(f"\n📊 Missing + Invalid Values Reduction:")
    for field_name in FIELD_CONFIG["strategy3_fields"].keys():
        output_field = f"imputed_{field_name}"
        if output_field in result_df_s3.columns:
            # Count original missing
            missing_before = df[field_name].isna().sum()
            # Count invalid values in original
            invalid_values = INVALID_VALUES_CONFIG.get(field_name, [])
            invalid_count = df[field_name].isin(invalid_values).sum()
            # Count after
            after = result_df_s3[output_field].isna().sum()
            total_before = missing_before + invalid_count
            print(f"  {field_name:25s}: {total_before} ({missing_before} missing + {invalid_count} invalid) → {after}")

## Step 4: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and effectiveness

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Field coverage per strategy
- Method comparison

**Strategy selection guide:**
- **Statistical**: Simple, fast, preserves distribution
- **Advanced**: Better for sequential/temporal data
- **Invalid**: Essential for data quality issues

**Note:** Fastest strategy isn't always best - consider data characteristics

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Statistical):      {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Advanced):         {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Invalid Handling): {elapsed_s3:6.2f}s")
print(f"  Total:                         {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print(f"\n📋 Field Coverage:")
print(f"  Strategy 1: {len(FIELD_CONFIG['strategy1_fields'])} fields")
print(f"  Strategy 2: {len(FIELD_CONFIG['strategy2_fields'])} fields")
print(f"  Strategy 3: {len(FIELD_CONFIG['strategy3_fields'])} fields")

print(f"\n🎯 Methods Used:")
print(f"  Strategy 1: {', '.join(set([v['imputation_strategy'] for v in FIELD_CONFIG['strategy1_fields'].values()]))}")
print(f"  Strategy 2: {', '.join(set([v['imputation_strategy'] for v in FIELD_CONFIG['strategy2_fields'].values()]))}")
print(f"  Strategy 3: {', '.join(set([v['imputation_strategy'] for v in FIELD_CONFIG['strategy3_fields'].values()]))} + invalid value handling")

## Step 5: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: Imputation counts, statistical comparisons, distribution impacts
2. **Output Data**: Sample of imputed records
3. **Visualizations**: Before/after distribution charts (latest batch only)

**Key metrics to check:**
- Imputed value counts and percentages
- Mean/median changes (should be minimal)
- Standard deviation changes (distribution preservation)
- Skewness and kurtosis impacts

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_statistical', 'Strategy 1: Statistical Imputation'),
    ('strategy2_advanced', 'Strategy 2: Advanced Methods'),
    ('strategy3_invalid', 'Strategy 3: Invalid Value Handling')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}\n")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Display imputed values
                if 'imputed_values' in metrics:
                    print(f"   Imputed Values:")
                    for field, stats in metrics['imputed_values'].items():
                        count = stats.get('count', 0)
                        percent = stats.get('percent', 0) * 100
                        print(f"     {field}: {count} values ({percent:.1f}%)")
                
                # Display statistical comparisons
                if 'statistical_comparisons' in metrics:
                    print(f"\n   Statistical Impact:")
                    for field, stats in metrics['statistical_comparisons'].items():
                        mean_before = stats.get('mean', {}).get('before')
                        mean_after = stats.get('mean', {}).get('after')
                        if mean_before is not None and mean_after is not None:
                            change = ((mean_after - mean_before) / mean_before * 100) if mean_before != 0 else 0
                            print(f"     {field} mean: {mean_before:.2f} → {mean_after:.2f} ({change:+.2f}%)")
                
                # Performance
                if 'execution_time_seconds' in metrics:
                    print(f"\n   Duration: {metrics['execution_time_seconds']:.4f}s")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:3]:  # Show first 3 only
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Calculate Data Quality Metrics

**How to use:**
- Run AFTER all strategies complete
- Calculates completeness improvements

**What you'll see:**
- Original data: missing value counts per field
- After imputation: zero missing values (target achieved)
- Completeness improvement ratios (e.g., 75% → 100%)
- Overall dataset quality score

**Quality targets:**
- Completeness ≥ 95%: Good data quality
- Completeness = 100%: Ideal state
- Distribution preserved: Critical for analysis

**Note:** Completeness = (total values - missing) / total values per field

In [ ]:
print("\n" + "=" * 80)
print("📊 DATA QUALITY METRICS")
print("=" * 80 + "\n")

def calculate_completeness(df, fields):
    """Calculate completeness percentage for given fields."""
    total_values = len(df) * len(fields)
    missing_values = sum(df[field].isna().sum() for field in fields if field in df.columns)
    completeness = (total_values - missing_values) / total_values * 100
    return completeness, missing_values

try:
    # Calculate for original data
    all_processed_fields = set()
    for strategy_fields in FIELD_CONFIG.values():
        all_processed_fields.update(strategy_fields.keys())
    
    completeness_orig, missing_orig = calculate_completeness(df, all_processed_fields)
    print(f"📊 ORIGINAL DATA:")
    print(f"   Fields analyzed: {len(all_processed_fields)}")
    print(f"   Total missing values: {missing_orig}")
    print(f"   Completeness: {completeness_orig:.2f}%")
    
    # Calculate for Strategy 3 (most comprehensive - includes invalid value handling)
    if 'result_df_s3' in locals() and result_df_s3 is not None:
        imputed_fields = [f"imputed_{field}" for field in FIELD_CONFIG['strategy3_fields'].keys()]
        completeness_final, missing_final = calculate_completeness(result_df_s3, imputed_fields)
        
        print(f"\n✨ AFTER IMPUTATION (Strategy 3):")
        print(f"   Fields imputed: {len(imputed_fields)}")
        print(f"   Total missing values: {missing_final}")
        print(f"   Completeness: {completeness_final:.2f}%")
        print(f"   Improvement: {completeness_final - completeness_orig:+.2f} percentage points")
        
        # Field-level comparison
        print(f"\n📈 FIELD-LEVEL IMPROVEMENT:")
        print(f"{'Field':<25} {'Before':>10} {'After':>10} {'Improvement':>12}")
        print("-" * 60)
        for field in FIELD_CONFIG['strategy3_fields'].keys():
            imputed_field = f"imputed_{field}"
            if field in df.columns and imputed_field in result_df_s3.columns:
                before_pct = (1 - df[field].isna().sum() / len(df)) * 100
                after_pct = (1 - result_df_s3[imputed_field].isna().sum() / len(result_df_s3)) * 100
                improvement = after_pct - before_pct
                print(f"{field:<25} {before_pct:>9.1f}% {after_pct:>9.1f}% {improvement:>11.1f}%")
        
        print(f"\n🎯 Dataset completeness: {completeness_final:.1f}%")
    else:
        print("\n⚠️  Run all strategies first!")
        
except NameError:
    print("⚠️  Run Step 3 (Setup Environment) first!")

## Step 7: Export Final Data

**How to use:**
- Run AFTER all strategies complete
- Exports imputed datasets for each strategy

**What you'll see (per strategy):**
- Available columns list
- Export path
- Preview (first 5 rows)
- Statistics (missing value reduction)

**Output structure:**
```
advanced_output/
├── strategy1_statistical/
│   └── statistical_imputed_data.csv
├── strategy2_advanced/
│   └── advanced_imputed_data.csv
└── strategy3_invalid/
    └── invalid_imputed_data.csv
```

**Note:** Files include both original and imputed fields for comparison

In [ ]:
import os
from pathlib import Path

print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Base export directory
export_base_dir = task_dir
task_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Export directory: {task_dir}\n")

# ============================================================================
# STRATEGY 1: Statistical Imputation
# ============================================================================
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: STATISTICAL IMPUTATION")
    print("=" * 80)
    
    s1_dir = export_base_dir / 'strategy1_statistical'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    # Select columns to export
    export_cols_s1 = ['record_id']
    for field in FIELD_CONFIG['strategy1_fields'].keys():
        export_cols_s1.extend([field, f'imputed_{field}'])
    
    existing_cols_s1 = [col for col in export_cols_s1 if col in result_df_s1.columns]
    final_df_s1 = result_df_s1[existing_cols_s1].copy()
    
    # Save
    output_path_s1 = s1_dir / 'statistical_imputed_data.csv'
    try:
        final_df_s1.to_csv(output_path_s1, index=False)
        print(f"\n✅ Saved: {output_path_s1}")
        print(f"   Rows: {len(final_df_s1):,}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    # Preview
    print(f"\n📊 Preview:")
    print(final_df_s1.head())
    
    # Statistics
    print(f"\n📈 Imputation Statistics:")
    for field in FIELD_CONFIG['strategy1_fields'].keys():
        if field in df.columns and f'imputed_{field}' in final_df_s1.columns:
            before = df[field].isna().sum()
            after = final_df_s1[f'imputed_{field}'].isna().sum()
            print(f"   {field}: {before} → {after} missing")
else:
    print("\n⚠️  Strategy 1 data not available")

# ============================================================================
# STRATEGY 2: Advanced Methods
# ============================================================================
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: ADVANCED METHODS")
    print("=" * 80)
    
    s2_dir = export_base_dir / 'strategy2_advanced'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    export_cols_s2 = ['record_id']
    for field in FIELD_CONFIG['strategy2_fields'].keys():
        export_cols_s2.extend([field, f'imputed_{field}'])
    
    existing_cols_s2 = [col for col in export_cols_s2 if col in result_df_s2.columns]
    final_df_s2 = result_df_s2[existing_cols_s2].copy()
    
    output_path_s2 = s2_dir / 'advanced_imputed_data.csv'
    try:
        final_df_s2.to_csv(output_path_s2, index=False)
        print(f"\n✅ Saved: {output_path_s2}")
        print(f"   Rows: {len(final_df_s2):,}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    print(f"\n📊 Preview:")
    print(final_df_s2.head())
    
    print(f"\n📈 Imputation Statistics:")
    for field in FIELD_CONFIG['strategy2_fields'].keys():
        if field in df.columns and f'imputed_{field}' in final_df_s2.columns:
            before = df[field].isna().sum()
            after = final_df_s2[f'imputed_{field}'].isna().sum()
            print(f"   {field}: {before} → {after} missing")
else:
    print("\n⚠️  Strategy 2 data not available")

# ============================================================================
# STRATEGY 3: Invalid Value Handling
# ============================================================================
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: INVALID VALUE HANDLING")
    print("=" * 80)
    
    s3_dir = export_base_dir / 'strategy3_invalid'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    export_cols_s3 = ['record_id']
    for field in FIELD_CONFIG['strategy3_fields'].keys():
        export_cols_s3.extend([field, f'imputed_{field}'])
    
    existing_cols_s3 = [col for col in export_cols_s3 if col in result_df_s3.columns]
    final_df_s3 = result_df_s3[existing_cols_s3].copy()
    
    output_path_s3 = s3_dir / 'invalid_imputed_data.csv'
    try:
        final_df_s3.to_csv(output_path_s3, index=False)
        print(f"\n✅ Saved: {output_path_s3}")
        print(f"   Rows: {len(final_df_s3):,}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    print(f"\n📊 Preview:")
    print(final_df_s3.head())
    
    print(f"\n📈 Imputation Statistics (including invalid values):")
    for field in FIELD_CONFIG['strategy3_fields'].keys():
        if field in df.columns and f'imputed_{field}' in final_df_s3.columns:
            missing_before = df[field].isna().sum()
            invalid_values = INVALID_VALUES_CONFIG.get(field, [])
            invalid_count = df[field].isin(invalid_values).sum()
            total_before = missing_before + invalid_count
            after = final_df_s3[f'imputed_{field}'].isna().sum()
            print(f"   {field}: {total_before} ({missing_before} missing + {invalid_count} invalid) → {after}")
else:
    print("\n⚠️  Strategy 3 data not available")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 Files saved to: {export_base_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 imputation strategies implemented and compared
- ✅ Data quality metrics calculated (completeness)
- ✅ Performance benchmarking completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Statistical methods (mean/median) best for normally distributed data
- Interpolation excels for sequential/time-series data
- Invalid value handling essential for data quality
- All methods preserve distribution characteristics

**Strategy recommendations:**
- **Use Strategy 1 (Statistical)** when: Data is normally distributed, simple and fast needed
- **Use Strategy 2 (Advanced)** when: Sequential patterns exist, temporal data, categorical modes
- **Use Strategy 3 (Invalid)** when: Known data quality issues, sentinel values present, cleaning required

**Best practices:**
- Always check distribution before/after imputation
- Validate mean/median changes are minimal (<5%)
- Document invalid value assumptions
- Test multiple strategies on sample data

**Next steps:**
- Test with your own datasets
- Tune methods for optimal results
- Deploy to production pipeline
- Monitor data quality metrics

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)
- 📊 [Data Quality Guide](../../docs/data_quality.md)